In [1]:
# 1. Import libraries and configure paths

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = PROJECT_ROOT / "data" / "splits"

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [2]:
# 2. Load train, validation, and test datasets

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
validation_df = pd.read_csv(VALIDATION_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining label distribution:")
print(train_df["label"].value_counts())

Training shape: (60957, 32)
Validation shape: (7618, 32)
Test shape: (7618, 32)

Training label distribution:
label
0    49866
1    11091
Name: count, dtype: int64


In [3]:
# 3. Create a sample corpus for Bag of Words

sample_messages = [
    "bank account verification",
    "bank account blocked",
    "money transfer completed"
]

sample_df = pd.DataFrame({
    "document": ["D1", "D2", "D3"],
    "text": sample_messages
})

display(sample_df)

,document,text
0,D1,bank account verification
1,D2,bank account blocked
2,D3,money transfer completed


In [4]:
# 4. Extract the vocabulary from sample documents

vocabulary = sorted(set(
    word
    for message in sample_messages
    for word in message.split()
))

print("Vocabulary size:", len(vocabulary))
print("Vocabulary:", vocabulary)

Vocabulary size: 7
Vocabulary: ['account', 'bank', 'blocked', 'completed', 'money', 'transfer', 'verification']


In [5]:
# 5. Generate the Bag of Words matrix

bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(sample_messages)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=sample_df["document"]
)

display(bow_df)

,account,bank,blocked,completed,money,transfer,verification
document,,,,,,,
D1,1,1,0,0,0,0,1
D2,1,1,1,0,0,0,0
D3,0,0,0,1,1,1,0


In [7]:
# 6. Demonstrate that Bag of Words ignores word order

order_messages = [
    "bank account",
    "account bank"
]

order_matrix = bow_vectorizer.transform(order_messages)

order_df = pd.DataFrame(
    order_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=order_messages
)

display(order_df[["account", "bank"]])

,account,bank
bank account,1,1
account bank,1,1


In [8]:
# 7. Demonstrate repeated word counting

repeated_messages = [
    "bank bank account",
    "account money money"
]

repeated_matrix = bow_vectorizer.transform(repeated_messages)

repeated_df = pd.DataFrame(
    repeated_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=["D1", "D2"]
)

display(repeated_df)

,account,bank,blocked,completed,money,transfer,verification
D1,1,2,0,0,0,0,0
D2,1,0,0,0,2,0,0


In [9]:
# 8. Calculate sparsity of the sample BoW matrix

total_elements = bow_matrix.shape[0] * bow_matrix.shape[1]
nonzero_elements = bow_matrix.nnz
zero_elements = total_elements - nonzero_elements

sparsity = (zero_elements / total_elements) * 100

print("Matrix shape:", bow_matrix.shape)
print("Total elements:", total_elements)
print("Non-zero elements:", nonzero_elements)
print("Zero elements:", zero_elements)
print(f"Sparsity: {sparsity:.2f}%")

Matrix shape: (3, 7)
Total elements: 21
Non-zero elements: 9
Zero elements: 12
Sparsity: 57.14%


In [10]:
# 9. Apply Bag of Words to the real training dataset

real_bow_vectorizer = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 1)
)

X_train_bow = real_bow_vectorizer.fit_transform(
    train_df["normalized_text"].fillna("")
)

print("Document-term matrix shape:", X_train_bow.shape)
print("Vocabulary size:", len(real_bow_vectorizer.vocabulary_))
print("Non-zero elements:", X_train_bow.nnz)

Document-term matrix shape: (60957, 10000)
Vocabulary size: 10000
Non-zero elements: 4698934


In [11]:
# 10. Calculate sparsity of the real BoW matrix

total_elements = X_train_bow.shape[0] * X_train_bow.shape[1]
nonzero_elements = X_train_bow.nnz

sparsity = (
    (total_elements - nonzero_elements) / total_elements
) * 100

print(f"Total elements: {total_elements:,}")
print(f"Non-zero elements: {nonzero_elements:,}")
print(f"Zero elements: {total_elements - nonzero_elements:,}")
print(f"Sparsity: {sparsity:.2f}%")

Total elements: 609,570,000
Non-zero elements: 4,698,934
Zero elements: 604,871,066
Sparsity: 99.23%


In [12]:
# 11. Inspect the vocabulary learned by BoW

bow_vocabulary = real_bow_vectorizer.get_feature_names_out()

print("Total vocabulary terms:", len(bow_vocabulary))
print("\nFirst 50 vocabulary terms:")
print(bow_vocabulary[:50])

Total vocabulary terms: 10000

First 50 vocabulary terms:
['00' '000' '0000' '00000' '00000000' '000000000' '00000million' '001'
 '002' '003' '005' '00pm' '01' '0100' '02' '0200' '02139' '03' '04' '0400'
 '05' '0500' '06' '0600' '06033' '07' '0700' '08' '09' '0d' '0x' '10'
 '100' '1000' '10000' '100000' '10019' '10019help' '1002' '10020' '10022'
 '100k' '101' '102' '1024' '103' '104' '105' '106' '107']


In [13]:
# 12. Discover the full vocabulary without a feature limit

full_bow_vectorizer = CountVectorizer()

full_bow_vectorizer.fit(
    train_df["normalized_text"].fillna("")
)

full_vocabulary = full_bow_vectorizer.get_feature_names_out()

print("Full vocabulary size:", len(full_vocabulary))
print("\nFirst 50 vocabulary terms:")
print(full_vocabulary[:50])

Full vocabulary size: 115436

First 50 vocabulary terms:
['00' '000' '0000' '00000' '000000' '0000000' '00000000' '000000000'
 '00000000000000' '00000000000000000000000000000000' '0000000000001'
 '000000000001' '000000000445' '00000001' '0000000149011611938477'
 '00000003' '000000064' '00000010' '000000128' '000000196' '00000049'
 '0000005' '0000006hz' '000000e' '000000eb' '000000i' '000000t0000z'
 '000001' '0000010' '00000100' '00000101' '00000111' '00000201' '00000354'
 '0000040' '000005' '00000eur' '00000kt' '00000million' '00001' '00001000'
 '00002' '0000211' '00003071548037755724105' '00003759' '00004' '000055'
 '00006' '000060' '000064000']


In [14]:
# 13. Generate the full-vocabulary BoW matrix

full_bow_matrix = full_bow_vectorizer.transform(
    train_df["normalized_text"].fillna("")
)

print("Document-term matrix shape:", full_bow_matrix.shape)
print("Vocabulary size:", len(full_bow_vectorizer.vocabulary_))
print("Non-zero elements:", full_bow_matrix.nnz)
print("Matrix data type:", full_bow_matrix.dtype)

Document-term matrix shape: (60957, 115436)
Vocabulary size: 115436
Non-zero elements: 5052694
Matrix data type: int64


In [15]:
# 14. Analyze numeric and alphanumeric vocabulary terms

import re

pure_numeric = [
    word for word in full_vocabulary
    if re.fullmatch(r"\d+", word)
]

alphanumeric = [
    word for word in full_vocabulary
    if re.search(r"\d", word)
]

print("Full vocabulary:", len(full_vocabulary))
print("Pure numeric tokens:", len(pure_numeric))
print("Tokens containing digits:", len(alphanumeric))

print(
    f"Pure numeric percentage: "
    f"{len(pure_numeric) / len(full_vocabulary) * 100:.2f}%"
)

print(
    f"Tokens containing digits percentage: "
    f"{len(alphanumeric) / len(full_vocabulary) * 100:.2f}%"
)

Full vocabulary: 115436
Pure numeric tokens: 7933
Tokens containing digits: 19762
Pure numeric percentage: 6.87%
Tokens containing digits percentage: 17.12%
